# 📚 Fiche de Révision Ultime : Machine Learning IMDS (V2 Complète)

Cette fiche rassemble l'intégralité des fonctions clés de tes TPs, de la régression linéaire jusqu'à la PCA et K-Means (TP8), optimisée pour une utilisation sous VS Code.

## 1. Importations et Préparation des Données

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.io import loadmat
from scipy import optimize
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle

In [ ]:
def load_and_prep_csv(filename):
    df = pd.read_csv(filename)
    # Encodage si présence de texte (ex: 'gender')
    if 'gender' in df.columns:
        le = LabelEncoder()
        df['gender'] = le.fit_transform(df['gender'])
    
    data = shuffle(np.array(df))
    X = data[:, :-1]
    y = data[:, -1]
    return X, y

def split_data(X, y, train_ratio=0.6, val_ratio=0.2):
    m = X.shape[0]
    s1, s2 = int(train_ratio * m), int((train_ratio + val_ratio) * m)
    return X[:s1], y[:s1], X[s1:s2], y[s1:s2], X[s2:], y[s2:]

def add_bias(X):
    return np.concatenate([np.ones((X.shape[0], 1)), X], axis=1)

## 2. Métriques d'Évaluation et Prédictions
Indispensable pour répondre aux questions d'examen sur les performances du modèle.

In [ ]:
def predict_logistic(theta, X):
    # Retourne 1 si proba >= 0.5, sinon 0
    return (sigmoid(np.dot(X, theta)) >= 0.5).astype(int)

def accuracy(y_pred, y_true):
    return np.mean(y_pred == y_true) * 100

def mse(y_pred, y_true):
    return np.mean(np.square(y_pred - y_true))

## 3. Régression Linéaire & Logistique (Rappels rapides)

In [ ]:
def linearRegCostFunction(X, y, theta, lambda_=0.0):
    m = y.size
    h = np.dot(X, theta)
    J = (1.0 / (2 * m)) * np.sum(np.square(h - y)) + (lambda_ / (2 * m)) * np.sum(np.square(theta[1:]))
    grad = (1.0 / m) * np.dot(X.T, (h - y))
    grad[1:] += (lambda_ / m) * theta[1:]
    return J, grad

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def lrCostFunction(theta, X, y, lambda_):
    m = y.size
    h = sigmoid(np.dot(X, theta))
    eps = 1e-5
    J = (1.0 / m) * np.sum(-y * np.log(h + eps) - (1 - y) * np.log(1 - h + eps)) 
    J += (lambda_ / (2 * m)) * np.sum(np.square(theta[1:]))
    grad = (1.0 / m) * np.dot(X.T, (h - y))
    grad[1:] += (lambda_ / m) * theta[1:]
    return J, grad

## 4. Apprentissage Non Supervisé : K-Means (TP8)
**Objectif :** Grouper les données en K clusters.
**Dimensions :** `centroids` est de taille (K, n), `idx` est de taille (m,).

In [ ]:
def findClosestCentroids(X, centroids):
    m = X.shape[0]
    idx = np.zeros(m, dtype=int)
    for i in range(m):
        # Distance Euclidienne au carré entre un point X[i] et tous les centroides
        distances = np.sum(np.square(X[i] - centroids), axis=1)
        idx[i] = np.argmin(distances)
    return idx

def computeCentroids(X, idx, K):
    m, n = X.shape
    centroids = np.zeros((K, n))
    for k in range(K):
        # Moyenne des points assignés au cluster k
        centroids[k, :] = np.mean(X[idx == k, :], axis=0)
    return centroids

def runKMeans(X, initial_centroids, max_iters):
    m, n = X.shape
    K = initial_centroids.shape[0]
    centroids = initial_centroids
    idx = np.zeros(m)
    for i in range(max_iters):
        idx = findClosestCentroids(X, centroids)
        centroids = computeCentroids(X, idx, K)
    return centroids, idx

def kMeansInitCentroids(X, K):
    # Initialisation aléatoire : on prend K points au hasard dans X
    rand_indices = np.random.permutation(X.shape[0])
    return X[rand_indices[:K], :]

## 5. Réduction de Dimension : PCA (TP8)
**Objectif :** Réduire le nombre de features (de n à K) tout en gardant un maximum de variance.
**ATTENTION :** Toujours normaliser les données (`featureNormalize`) avant d'appliquer la PCA !

In [ ]:
def featureNormalize(X):
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0, ddof=1)
    X_norm = (X - mu) / sigma
    return X_norm, mu, sigma

def pca(X):
    m, n = X.shape
    # 1. Matrice de covariance
    Sigma = (1.0 / m) * np.dot(X.T, X)
    # 2. Décomposition en valeurs singulières (SVD)
    U, S, V = np.linalg.svd(Sigma)
    # U contient les composantes principales
    return U, S

def projectData(X, U, K):
    # Projection des données sur les K premières composantes
    U_reduce = U[:, :K]
    return np.dot(X, U_reduce)

def recoverData(Z, U, K):
    # Reconstruction approximative des données originales (pour visualisation)
    U_reduce = U[:, :K]
    return np.dot(Z, U_reduce.T)

## 6. Ajout de Features (Polynomial Mapping)
Utile si le modèle sous-apprend (High Bias) et que la frontière de décision n'est pas linéaire.

In [ ]:
def polyFeatures(X, p):
    # Prend un vecteur X (m, 1) et retourne une matrice (m, p) [X, X^2, ..., X^p]
    X_poly = np.zeros((X.shape[0], p))
    for i in range(1, p + 1):
        X_poly[:, i-1] = np.power(X[:, 0], i)
    return X_poly